# 01 - Data and financial concepts

**Question of the project:** does a change in public attention to a company (Wikipedia pageviews) help
predict the *realized volatility* of a related ETF, once we already use everything that past volatility tells us?

This notebook introduces the data and every financial/statistical concept used later.

## Glossary (read this first)

| Concept | Meaning |
|---|---|
| **ETF** (Exchange-Traded Fund) | A fund that trades on an exchange like a stock and holds a basket of assets. QQQ holds the Nasdaq-100 (large tech stocks), SPY the S&P 500, XLE energy stocks, XLF financials, XLK technology, GLD physical gold. We use ETFs instead of single stocks because they are very liquid and diversified. |
| **Adjusted close** | Closing price corrected for dividends and splits, so that price changes reflect only the investor's return. Without it a dividend looks like a crash. |
| **Return / log return** | $r_t=\ln(P_t/P_{t-1})$. Log returns add up over time and are roughly stationary, unlike prices. |
| **Volatility** | How much returns fluctuate: the standard deviation of returns. It is *risk*: it drives option prices, margin requirements, position sizing and VaR. |
| **Realized volatility (RV)** | Volatility *measured* from past returns (here: std of the last 5 daily log returns). Opposite of **implied volatility**, which is the market's *expectation* backed out of option prices. |
| **Volatility clustering** | Calm periods follow calm periods, turbulent follow turbulent: volatility is strongly autocorrelated (Mandelbrot, 1963; Engle's ARCH, 1982). This is the main reason forecasting it is easier than forecasting returns. |
| **Stationarity** | A series whose mean/variance/autocorrelation do not drift over time. Regressions on non-stationary series (like prices, $I(1)$) give *spurious* relations, so we model returns and log-changes. |
| **ADF test** | Augmented Dickey-Fuller: H0 = the series has a unit root (non-stationary). A small p-value rejects H0, i.e. supports stationarity. |
| **Pageviews as attention** | Daily visits to a Wikipedia article. A spike may mean news or public interest *before or while* markets react (Moat et al. 2013; Preis et al. 2013 used this idea for trading). |
| **Look-ahead bias** | Using information not yet available at forecast time. The most common way backtests lie. We use only data dated $t-1$ or earlier. |

In [ ]:
import os, sys, warnings
os.chdir(os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd())
sys.path.insert(0, os.getcwd())
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.api as sm
from src import model_pipeline as mp
plt.rcParams.update({"figure.figsize": (9, 3.6), "axes.grid": True, "grid.alpha": .3})
START, END = "2016-01-01", "2026-10-31"

## 1. Prices and returns (QQQ)

In [ ]:
px = mp.fetch_prices("QQQ", START, END)
r = np.log(px).diff().dropna()
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
px.plot(ax=ax[0], title="QQQ adjusted close (non-stationary, trending)")
r.plot(ax=ax[1], lw=.6, title="QQQ daily log return (stationary, but with calm/turbulent regimes)")
plt.tight_layout(); plt.show()
print(f"{len(r)} trading days, {r.index[0].date()} -> {r.index[-1].date()}")

Prices drift upward (non-stationary); returns fluctuate around zero. Note the **2020 COVID crash** and the
**2022 bear market**: returns are much larger there. That is volatility clustering, visible by eye.

## 2. Realized volatility and Wikipedia attention

In [ ]:
rv5 = r.rolling(5).std()
views = mp.fetch_views("Nvidia", START, END)
fig, ax = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
rv5.plot(ax=ax[0], lw=.8, title="QQQ 5-day realized volatility (daily units)")
views.plot(ax=ax[1], lw=.8, logy=True, title="Wikipedia daily pageviews, article 'Nvidia' (log scale)")
plt.tight_layout(); plt.show()

Volatility spikes in 2020 and 2022. Nvidia pageviews have a strong *secular trend* (the company became famous),
which is why we do **not** use raw views but their *daily log-change*: it measures a sudden jump in interest
and removes the slow popularity trend.

## 3. Volatility clustering: autocorrelation

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf
fig, ax = plt.subplots(1, 2, figsize=(12, 3.4))
plot_acf(r, lags=40, ax=ax[0], title="ACF of returns (about zero: returns are hard to predict)")
plot_acf(np.log(r.abs() + 1e-6), lags=40, ax=ax[1], title="ACF of log |return| (persistent volatility)")
plt.tight_layout(); plt.show()

Returns are nearly uncorrelated, but their *size* is highly persistent. Consequence for this project: a model that only
looks at past volatility will already do very well, so **attention has to add information beyond persistence** to matter.
That is why the baseline in notebook 02 is the real benchmark.

## 4. Stationarity (ADF) on the series actually used

The model target is $y_t=\ln \mathrm{RV}^{(5)}_t$ and the attention feature is $\Delta\ln(\text{views}_t)$.

In [ ]:
from statsmodels.tsa.stattools import adfuller
X, y = mp.build("QQQ", "Nvidia", START, END)
rows = {
    "QQQ price (log)": np.log(px),
    "QQQ log return": r,
    "target: log RV5": y,
    "feature: dlog views (lag 1)": X["views_l1"],
}
out = {}
for k, v in rows.items():
    a = adfuller(v.dropna(), result_object=True)
    out[k] = (a.statistic, a.pvalue)
pd.DataFrame(out, index=["ADF stat", "p-value"]).T

The log price **cannot reject** a unit root (large p-value): it is $I(1)$. Returns, log-volatility and log-changes of views
reject it ($p<0.001$): they are safe to use in a regression.

## 5. A first look: does attention move with volatility?

Cross-correlation between the change in views and log RV (minus its 60-day mean) at different lags. Lag $k>0$ means *views lead volatility by k days*.
Contemporaneous correlation (lag 0) is **not** usable for forecasting - it is exactly the look-ahead trap.

In [ ]:
dv = np.log(views.reindex(rv5.index)).diff()
lv = np.log(rv5)
ccf = {k: dv.shift(k).corr(lv - lv.rolling(60).mean()) for k in range(-5, 11)}
pd.Series(ccf).plot(kind="bar", title="corr(dlog views_{t-k}, log RV_t minus 60d mean) - QQQ / Nvidia", figsize=(9, 3.4))
plt.xlabel("k (days views lead volatility)"); plt.show()

Whatever correlation exists is small. Notebook 02 tests it properly: does it survive *after* conditioning on past
volatility, and *out of sample*?